# Part C: Text Classification - Step 4: From-Scratch Classifier in NumPy

This notebook implements a **Multinomial (Softmax) Logistic Regression** classifier **from scratch using NumPy only** (without `scikit-learn` for model training or inference), trained with Mini-Batch / Full-Batch Gradient Descent and $L_2$ regularization.

We then evaluate and compare the NumPy from-scratch implementation against `sklearn.linear_model.LogisticRegression` on the exact same train/test split and TF-IDF features, comparing **Accuracy**, **Macro F1-score**, **training time**, and **convergence behavior**.

In [ ]:
import os
import re
import string
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

%matplotlib inline
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load & Preprocess Text Dataset

We use the same cleaning function, stratified 70/15/15 split (`random_state=42`), and `TfidfVectorizer(max_features=5000, stop_words='english')` as `01_text_preprocessing_logistic.ipynb`. If `../data/bbc_news.csv` is not present locally, a small synthetic multi-class news corpus is generated so the notebook remains executable end-to-end.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>+", "", text)
    text = re.sub(r"[%s]" % re.escape(string.punctuation), "", text)
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\w*\d\w*", "", text)
    return text.strip()

data_path = "../data/bbc_news.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    df["clean_text"] = df["text"].apply(clean_text)
    print(f"Loaded BBC News dataset: {df.shape}")
else:
    # Fallback synthetic corpus if bbc_news.csv is not yet placed in ../data/
    rng = np.random.default_rng(RANDOM_STATE)
    templates = {
        "business": ["stock market shares profit company revenue economy bank growth investment"],
        "tech": ["software smartphone artificial intelligence data cloud network cyber security"],
        "sport": ["football match goal tournament champion player coach league stadium"],
        "politics": ["government election parliament minister policy vote tax debate"],
        "entertainment": ["film movie actor music album festival award cinema director"],
    }
    rows = []
    for _ in range(300):
        cat = rng.choice(list(templates.keys()))
        words = templates[cat][0].split()
        sample = " ".join(rng.choice(words, size=12, replace=True))
        rows.append({"text": sample, "category": cat})
    df = pd.DataFrame(rows)
    df["clean_text"] = df["text"].apply(clean_text)
    print("bbc_news.csv not found; using synthetic multi-class fallback corpus for demonstration.")

X_temp, X_test_raw, y_temp, y_test_str = train_test_split(
    df["clean_text"],
    df["category"],
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=df["category"],
)

X_train_raw, X_val_raw, y_train_str, y_val_str = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train_raw).toarray()
X_test_tfidf = vectorizer.transform(X_test_raw).toarray()

classes = np.unique(y_train_str)
class_to_idx = {c: i for i, c in enumerate(classes)}
y_train = np.array([class_to_idx[c] for c in y_train_str])
y_test = np.array([class_to_idx[c] for c in y_test_str])

print(f"Train shape: {X_train_tfidf.shape}, Test shape: {X_test_tfidf.shape}, Classes: {list(classes)}")

## 2. From-Scratch Multinomial Logistic Regression (NumPy Only)

We implement **Softmax Regression** using pure NumPy:
- **Forward pass:** $Z = XW + b$, $\hat{Y} = \text{softmax}(Z)$ with max-subtraction for numerical stability.
- **Loss function:** Cross-entropy loss with $L_2$ weight decay:
  $$\mathcal{L}(W, b) = -\frac{1}{N}\sum_{i=1}^N \sum_{k=1}^K y_{ik} \log(\hat{y}_{ik} + \epsilon) + \frac{\lambda}{2N} \|W\|_F^2$$
- **Optimization:** Gradient Descent updates $W \leftarrow W - \eta \nabla_W \mathcal{L}$ and $b \leftarrow b - \eta \nabla_b \mathcal{L}$.

In [ ]:
class NumpySoftmaxLogisticRegression:
    """Multinomial Logistic Regression trained with Gradient Descent using NumPy only."""

    def __init__(self, learning_rate=1.0, n_epochs=300, l2_reg=1e-4, random_state=42):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.l2_reg = l2_reg
        self.random_state = random_state
        self.W = None
        self.b = None
        self.loss_history = []

    @staticmethod
    def _softmax(logits):
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    @staticmethod
    def _one_hot(y, num_classes):
        one_hot = np.zeros((y.shape[0], num_classes), dtype=np.float64)
        one_hot[np.arange(y.shape[0]), y] = 1.0
        return one_hot

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape
        num_classes = int(np.max(y)) + 1

        self.W = rng.normal(0.0, 0.01, size=(n_features, num_classes))
        self.b = np.zeros((1, num_classes), dtype=np.float64)
        Y_true = self._one_hot(y, num_classes)
        self.loss_history = []

        for _ in range(self.n_epochs):
            logits = X @ self.W + self.b
            probs = self._softmax(logits)

            # Cross-entropy + L2 regularization loss
            ce_loss = -np.mean(np.sum(Y_true * np.log(probs + 1e-12), axis=1))
            reg_loss = (self.l2_reg / (2.0 * n_samples)) * np.sum(self.W ** 2)
            self.loss_history.append(ce_loss + reg_loss)

            # Gradients
            dZ = (probs - Y_true) / n_samples
            dW = (X.T @ dZ) + (self.l2_reg / n_samples) * self.W
            db = np.sum(dZ, axis=0, keepdims=True)

            # Parameter update
            self.W -= self.learning_rate * dW
            self.b -= self.learning_rate * db

        return self

    def predict_proba(self, X):
        logits = X @ self.W + self.b
        return self._softmax(logits)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

## 3. Train & Time Both Models (NumPy From-Scratch vs. scikit-learn)

In [ ]:
# 1. Train NumPy From-Scratch Logistic Regression
t0 = time.perf_counter()
numpy_clf = NumpySoftmaxLogisticRegression(
    learning_rate=2.0, n_epochs=400, l2_reg=1e-4, random_state=RANDOM_STATE
)
numpy_clf.fit(X_train_tfidf, y_train)
numpy_train_time = time.perf_counter() - t0
y_pred_numpy = numpy_clf.predict(X_test_tfidf)

# 2. Train scikit-learn Logistic Regression
t0 = time.perf_counter()
sklearn_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
sklearn_clf.fit(X_train_tfidf, y_train)
sklearn_train_time = time.perf_counter() - t0
y_pred_sklearn = sklearn_clf.predict(X_test_tfidf)

# Evaluate both
comparison_df = pd.DataFrame([
    {
        "Implementation": "NumPy From-Scratch (GD)",
        "Accuracy": accuracy_score(y_test, y_pred_numpy),
        "Macro F1": f1_score(y_test, y_pred_numpy, average="macro"),
        "Train Time (s)": numpy_train_time,
    },
    {
        "Implementation": "scikit-learn (L-BFGS)",
        "Accuracy": accuracy_score(y_test, y_pred_sklearn),
        "Macro F1": f1_score(y_test, y_pred_sklearn, average="macro"),
        "Train Time (s)": sklearn_train_time,
    },
])

display(comparison_df.style.format({"Accuracy": "{:.4f}", "Macro F1": "{:.4f}", "Train Time (s)": "{:.4f}"}))

## 4. Convergence Behavior & Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Convergence curve for NumPy Gradient Descent
axes[0].plot(numpy_clf.loss_history, color="royalblue", lw=2)
axes[0].set_title("NumPy Logistic Regression Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy + L2 Loss")

# Confusion Matrix: NumPy
sns.heatmap(confusion_matrix(y_test, y_pred_numpy), annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title("NumPy From-Scratch Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

# Confusion Matrix: scikit-learn
sns.heatmap(confusion_matrix(y_test, y_pred_sklearn), annot=True, fmt="d", cmap="Greens",
            xticklabels=classes, yticklabels=classes, ax=axes[2])
axes[2].set_title("scikit-learn Confusion Matrix")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")

plt.tight_layout()
plt.show()

print("NumPy Classification Report:\n", classification_report(y_test, y_pred_numpy, target_names=classes))
print("scikit-learn Classification Report:\n", classification_report(y_test, y_pred_sklearn, target_names=classes))

## 5. Discussion: NumPy From-Scratch vs. scikit-learn

1. **Optimization Algorithm & Convergence:**
   - Our NumPy implementation uses **first-order Batch Gradient Descent** with a fixed learning rate (`learning_rate=2.0`), which takes hundreds of iterations to descend the convex cross-entropy surface smoothly.
   - `scikit-learn`'s `LogisticRegression` defaults to **L-BFGS**, a quasi-Newton second-order optimization method that approximates the inverse Hessian matrix, enabling much faster convergence in fewer iterations.

2. **Sparse vs. Dense Matrix Operations:**
   - TF-IDF matrices are typically very sparse (mostly zeros). `scikit-learn` operates directly on `scipy.sparse` CSR matrices in compiled Cython/C, keeping memory usage low and matrix multiplications fast.
   - Our pure NumPy implementation converts the TF-IDF matrix to a dense array (`.toarray()`), which is straightforward and educational but uses more memory and computes multiplications over zero entries.

3. **Accuracy & Regularization:**
   - Because multinomial logistic regression has a convex objective, both implementations converge toward similar decision boundaries when learning rate and regularization (`l2_reg` vs. `C=1.0`) are appropriately tuned, typically yielding comparable Accuracy and Macro F1 scores.